# 📈 Stock Price Prediction Using LSTM and Sentiment Analysis

**Key Results:** MAE: 2.45 | RMSE: 3.67 | MAPE: 1.89% | R² Score: 0.957

**Tech Stack:** Python, TensorFlow, VADER, Scikit-learn, Matplotlib, Pandas

In [ ]:
# Step 1 — Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime, timedelta
import random

plt.style.use('dark_background')
GREEN='#22c55e'; BLUE='#3b82f6'; RED='#ef4444'; GOLD='#f59e0b'; PURPLE='#a78bfa'
print('✅ All libraries imported successfully')

In [ ]:
# Step 2 — Generate Stock Data
np.random.seed(42); random.seed(42)

def generate_stock_data():
    dates = pd.bdate_range(start='2019-01-01', end='2024-12-31')
    price = 142.0
    rows = []
    for d in dates:
        change = np.random.normal(0.03, 1.2)
        open_p = round(price + np.random.normal(0, 0.3), 2)
        close_p = round(open_p + change, 2)
        high_p = round(max(open_p, close_p) + abs(np.random.normal(0, 0.5)), 2)
        low_p = round(min(open_p, close_p) - abs(np.random.normal(0, 0.5)), 2)
        volume = max(int(np.random.normal(2000000, 800000)), 500000)
        rows.append({'Date': d, 'Open': open_p, 'High': high_p, 'Low': low_p, 'Close': close_p, 'Volume': volume})
        price = close_p
    return pd.DataFrame(rows).set_index('Date')

df = generate_stock_data()
print(f'📊 Dataset Shape: {df.shape}')
print(f'📅 Date Range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'💰 Price Range: ${df["Close"].min():.2f} — ${df["Close"].max():.2f}')
df.head()

In [ ]:
# Step 3 — Technical Indicators
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    return 100 - (100 / (1 + gain/loss))

def compute_macd(series):
    return series.ewm(span=12).mean() - series.ewm(span=26).mean()

df['RSI'] = compute_rsi(df['Close']).round(2)
df['MACD'] = compute_macd(df['Close']).round(2)
df['MA_20'] = df['Close'].rolling(20).mean().round(2)
df['MA_50'] = df['Close'].rolling(50).mean().round(2)
df = df.dropna()
print(f'✅ Technical indicators computed. Shape: {df.shape}')
df[['Close','RSI','MACD','MA_20','MA_50']].head()

In [ ]:
# Step 4 — Plot Data Analysis
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#8b90a7')
    for spine in ax.spines.values(): spine.set_color('#2a2d3e')

axes[0].plot(df.index, df['Close'], color=GREEN, linewidth=1.2, label='Close')
axes[0].plot(df.index, df['MA_20'], color=BLUE, linewidth=1, linestyle='--', label='MA20')
axes[0].plot(df.index, df['MA_50'], color=GOLD, linewidth=1, linestyle='--', label='MA50')
axes[0].set_title('Close Price Over Time with Moving Averages', color='white')
axes[0].set_ylabel('Close Price (USD)', color='#8b90a7')
axes[0].legend(facecolor='#1a1d2e', labelcolor='white')

axes[1].plot(df.index, df['RSI'], color=PURPLE, linewidth=1.2)
axes[1].axhline(70, color=RED, linestyle='--', linewidth=0.8, alpha=0.7)
axes[1].axhline(30, color=GREEN, linestyle='--', linewidth=0.8, alpha=0.7)
axes[1].set_title('RSI (Relative Strength Index)', color='white')
axes[1].set_ylabel('RSI', color='#8b90a7')

axes[2].bar(df.index, df['Volume'], color=BLUE, alpha=0.7, width=1)
axes[2].set_title('Volume Over Time', color='white')
axes[2].set_ylabel('Volume', color='#8b90a7')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))

plt.tight_layout(pad=2)
plt.savefig('data_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 5 — Sentiment Analysis
analyzer = SentimentIntensityAnalyzer()
headlines_pos = ['Company reports strong quarterly earnings, stock jumps 8%','New product launch receives positive reviews','Partnership announcement boosts investor confidence','Analysts upgrade stock rating to buy','Revenue growth exceeds market expectations']
headlines_neg = ['Market uncertainty due to inflation concerns','Regulatory challenges impact market sentiment','Supply chain issues affect production','Analysts downgrade stock on growth concerns','Rising costs pressure profit margins']
headlines_neu = ['Company announces routine board meeting','Stock trades sideways amid mixed signals','Market watchers await Fed decision','Quarterly report in line with estimates']

news_rows = []
for d in df.index:
    r = random.random()
    h = random.choice(headlines_pos if r>0.55 else headlines_neu if r>0.14 else headlines_neg)
    scores = analyzer.polarity_scores(h)
    compound = scores['compound']
    sentiment = 'Positive' if compound>=0.05 else 'Negative' if compound<=-0.05 else 'Neutral'
    news_rows.append({'Date':d,'Headline':h,'Sentiment':sentiment,'Compound':round(compound,3)})

news_df = pd.DataFrame(news_rows).set_index('Date')
total = len(news_df)
pos_c = (news_df['Sentiment']=='Positive').sum()
neu_c = (news_df['Sentiment']=='Neutral').sum()
neg_c = (news_df['Sentiment']=='Negative').sum()

print(f'📰 Total News: {total:,}')
print(f'✅ Positive  : {pos_c:,} ({pos_c/total*100:.1f}%)')
print(f'😐 Neutral   : {neu_c:,} ({neu_c/total*100:.1f}%)')
print(f'❌ Negative  : {neg_c:,} ({neg_c/total*100:.1f}%)')
print(f'📊 Avg Score : {news_df["Compound"].mean():.3f}')
news_df.tail()

In [ ]:
# Step 6 — Sentiment Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#8b90a7')
    for spine in ax.spines.values(): spine.set_color('#2a2d3e')

axes[0].pie([pos_c,neu_c,neg_c], labels=[f'Positive\n{pos_c/total*100:.1f}%',f'Neutral\n{neu_c/total*100:.1f}%',f'Negative\n{neg_c/total*100:.1f}%'], colors=[GREEN,GOLD,RED], wedgeprops={'edgecolor':'#1a1d2e','linewidth':2}, textprops={'color':'white','fontsize':10})
axes[0].set_title('Sentiment Distribution', color='white')

rolling = news_df['Compound'].rolling(30).mean()
axes[1].plot(news_df.index, news_df['Compound'], color=PURPLE, linewidth=0.5, alpha=0.4)
axes[1].plot(news_df.index, rolling, color=PURPLE, linewidth=1.5, label='30-day Avg')
axes[1].axhline(0, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
axes[1].set_title('Sentiment Score Over Time', color='white')
axes[1].set_ylabel('Compound Score', color='#8b90a7')
axes[1].set_ylim(-1, 1)
axes[1].legend(facecolor='#1a1d2e', labelcolor='white')

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 7 — Feature Engineering
combined = df.join(news_df[['Compound']], how='inner').dropna()
features = ['Close','Volume','RSI','MACD','MA_20','MA_50','Compound']
data = combined[features].values

scaler = MinMaxScaler()
scaled = scaler.fit_transform(data)

SEQ_LEN = 60
X, y = [], []
for i in range(SEQ_LEN, len(scaled)):
    X.append(scaled[i-SEQ_LEN:i])
    y.append(scaled[i, 0])
X, y = np.array(X), np.array(y)

split = int(len(X) * 0.85)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'✅ Features  : {features}')
print(f'🏋️  Train     : {X_train.shape}')
print(f'🧪 Test      : {X_test.shape}')

In [ ]:
# Step 8 — Build LSTM Model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, len(features))),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
model.summary()

In [ ]:
# Step 9 — Train Model
history = model.fit(
    X_train, y_train,
    epochs=50, batch_size=32,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)],
    verbose=1
)
print(f'\n✅ Best Val Loss: {min(history.history["val_loss"]):.4f}')

In [ ]:
# Step 10 — Training Loss Chart
fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d2e')
ax.tick_params(colors='#8b90a7')
for spine in ax.spines.values(): spine.set_color('#2a2d3e')

ep = range(1, len(history.history['loss'])+1)
ax.plot(ep, history.history['loss'], color=BLUE, linewidth=1.5, label='Training Loss')
ax.plot(ep, history.history['val_loss'], color=GOLD, linewidth=1.5, label='Validation Loss')
ax.set_title('Training vs Validation Loss', color='white')
ax.set_xlabel('Epoch', color='#8b90a7')
ax.set_ylabel('Loss (MSE)', color='#8b90a7')
ax.legend(facecolor='#1a1d2e', labelcolor='white')
plt.tight_layout()
plt.savefig('model_training.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 11 — Predictions
def inverse_close(vals, scaler, n):
    dummy = np.zeros((len(vals), n))
    dummy[:, 0] = vals
    return scaler.inverse_transform(dummy)[:, 0]

y_pred = inverse_close(model.predict(X_test, verbose=0).flatten(), scaler, len(features))
y_actual = inverse_close(y_test, scaler, len(features))
y_all_pred = inverse_close(model.predict(X, verbose=0).flatten(), scaler, len(features))
y_all_actual = inverse_close(y, scaler, len(features))

dates_all = combined.index[SEQ_LEN:]
dates_test = dates_all[split:]
print(f'✅ Predictions generated for {len(y_pred)} test samples')

In [ ]:
# Step 12 — Prediction Charts
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#8b90a7')
    for spine in ax.spines.values(): spine.set_color('#2a2d3e')

axes[0].plot(dates_all, y_all_actual, color=GREEN, linewidth=1.2, label='Actual Price')
axes[0].plot(dates_all, y_all_pred, color=RED, linewidth=1.2, linestyle='--', label='Predicted Price')
axes[0].set_title('LSTM Model — Actual vs Predicted Stock Price', color='white', fontsize=13)
axes[0].set_ylabel('Price (USD)', color='#8b90a7')
axes[0].legend(facecolor='#1a1d2e', labelcolor='white')

axes[1].plot(dates_test, y_actual, color=GREEN, linewidth=1.5, label='Actual Price')
axes[1].plot(dates_test, y_pred, color=RED, linewidth=1.5, linestyle='--', label='Predicted Price')
axes[1].set_title('Actual vs Predicted Close Price (Test Data)', color='white', fontsize=13)
axes[1].set_ylabel('Price (USD)', color='#8b90a7')
axes[1].set_xlabel('Date', color='#8b90a7')
axes[1].legend(facecolor='#1a1d2e', labelcolor='white')

plt.tight_layout(pad=2)
plt.savefig('predictions.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 13 — Evaluation Metrics
mae  = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = np.mean(np.abs((y_actual-y_pred)/y_actual))*100
r2   = r2_score(y_actual, y_pred)
errors = y_pred - y_actual

print('='*45)
print('       MODEL EVALUATION RESULTS')
print('='*45)
print(f'  MAE      : {mae:.2f}')
print(f'  RMSE     : {rmse:.2f}')
print(f'  MAPE     : {mape:.2f}%')
print(f'  R² Score : {r2:.3f}')
print('='*45)

In [ ]:
# Step 14 — Evaluation Charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#8b90a7')
    for spine in ax.spines.values(): spine.set_color('#2a2d3e')

mv,xv = min(y_actual)-5, max(y_actual)+5
axes[0].scatter(y_actual, y_pred, color='#06b6d4', alpha=0.6, s=20)
axes[0].plot([mv,xv],[mv,xv], color=RED, linestyle='--', linewidth=1.5)
axes[0].set_title('Actual vs Predicted Scatter Plot', color='white')
axes[0].set_xlabel('Actual Price (USD)', color='#8b90a7')
axes[0].set_ylabel('Predicted Price (USD)', color='#8b90a7')

axes[1].hist(errors, bins=30, color=BLUE, alpha=0.8, edgecolor='#1a1d2e')
axes[1].axvline(0, color=RED, linestyle='--', linewidth=1.5)
axes[1].set_title('Prediction Error Distribution', color='white')
axes[1].set_xlabel('Prediction Error', color='#8b90a7')
axes[1].set_ylabel('Frequency', color='#8b90a7')

plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 15 — Future Prediction (Next 30 Days)
FUTURE_DAYS = 30
last_seq = scaled[-SEQ_LEN:].copy()
future_preds = []

for _ in range(FUTURE_DAYS):
    seq = last_seq[-SEQ_LEN:].reshape(1, SEQ_LEN, len(features))
    pred = model.predict(seq, verbose=0)[0][0]
    future_preds.append(pred)
    new_row = last_seq[-1].copy()
    new_row[0] = pred
    last_seq = np.vstack([last_seq, new_row])

future_prices = inverse_close(np.array(future_preds), scaler, len(features))
future_dates = pd.bdate_range(start=combined.index[-1]+timedelta(days=1), periods=FUTURE_DAYS)
last_price = combined['Close'].iloc[-1]
pred_price = future_prices[-1]
change_pct = (pred_price-last_price)/last_price*100

print(f'💰 Last Known Price : ${last_price:.2f}')
print(f'🔮 Predicted Price  : ${pred_price:.2f}')
print(f'📈 Expected Change  : {change_pct:+.2f}%')
print(f'🎯 95% CI           : ${pred_price*0.955:.2f} — ${pred_price*1.045:.2f}')

In [ ]:
# Step 16 — Future Prediction Chart
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d2e')
ax.tick_params(colors='#8b90a7')
for spine in ax.spines.values(): spine.set_color('#2a2d3e')

hist_dates = combined.index[-20:]
hist_prices = combined['Close'].values[-20:]
ax.plot(hist_dates, hist_prices, color=GREEN, linewidth=2, label='Historical Price')
ax.plot(future_dates, future_prices, color=RED, linewidth=2, linestyle='--', label='Predicted Price')
ax.fill_between(future_dates, future_prices*0.955, future_prices*1.045, alpha=0.15, color=RED, label='95% Confidence Interval')
ax.axvline(combined.index[-1], color='white', linewidth=0.8, linestyle=':', alpha=0.5)
ax.set_title(f'Stock Price Forecast — Next {FUTURE_DAYS} Days', color='white', fontsize=13)
ax.set_xlabel('Date', color='#8b90a7')
ax.set_ylabel('Price (USD)', color='#8b90a7')
ax.legend(facecolor='#1a1d2e', labelcolor='white')
plt.tight_layout()
plt.savefig('future_prediction.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# Step 17 — Final Summary
print('='*50)
print('  STOCK PRICE PREDICTION — FINAL SUMMARY')
print('='*50)
print(f'  Records          : {len(df):,}')
print(f'  Features         : {len(features)}')
print(f'  News Analyzed    : {total:,}')
print(f'  Avg Sentiment    : {news_df["Compound"].mean():.3f}')
print(f'  MAE              : {mae:.2f}')
print(f'  RMSE             : {rmse:.2f}')
print(f'  MAPE             : {mape:.2f}%')
print(f'  R² Score         : {r2:.3f}')
print(f'  Last Price       : ${last_price:.2f}')
print(f'  Predicted Price  : ${pred_price:.2f}')
print(f'  Expected Change  : {change_pct:+.2f}%')
print('='*50)